# Conformer-ByteHum: Validation-First Retrieval Ablation

No training or checkpoint modification. Uses the original full-recording gallery
cache from the completed evaluator, plus resumable query embeddings and scores.
Validation uses all validation queries and only train/val gallery songs (255).
Test is evaluated only after a complete validation run locks a configuration.
The counterpart with 45 songs uses the SAME original recordings, not fragments.

The engine is `../experiments/retrieval_ablation.py`. CPU gallery tensors and
bounded similarity blocks keep this compatible with a 4 GB CUDA GPU. Embedding
DTW is an experimental temporal alignment, not a pitch-DTW baseline.


In [ ]:
from pathlib import Path
import sys
import json
import importlib.util
import pandas as pd

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / '02_stage2_humming_qbh').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Open this notebook from the project directory.')
STAGE2_ROOT = PROJECT_ROOT / '02_stage2_humming_qbh'
MODEL_LABEL = 'kaggle_v4'  # 'local_4gb' is also supported.
MAX_VALIDATION_QUERIES = None  # A subset cannot select a model or unlock test.
RUN_EMBEDDING_DTW = False  # First run the faster window-scoring experiment.
RUN_LOCKED_TEST = True

engine_path = STAGE2_ROOT / 'experiments' / 'retrieval_ablation.py'
spec = importlib.util.spec_from_file_location('retrieval_ablation', engine_path)
engine = importlib.util.module_from_spec(spec)
spec.loader.exec_module(engine)


## Validation: Full Original-Recording Gallery, No Test Songs


In [ ]:
validation_report, selection = engine.run(
    model=MODEL_LABEL, split='val', max_queries=MAX_VALIDATION_QUERIES,
    dtw=RUN_EMBEDDING_DTW)
display(pd.DataFrame(validation_report['metrics']).T)
print('Same-audio 45-song control:', validation_report['same_audio_heldout_gallery'])
if selection is not None:
    print('Locked configuration:', selection['winner'], selection['configuration'])
else:
    print('Subset validation: no selection has been written.')


## Test: Locked Configuration, 300 Original Recordings


In [ ]:
if RUN_LOCKED_TEST:
    if selection is None:
        raise RuntimeError('Complete all validation queries before running test.')
    test_report, _ = engine.run(model=MODEL_LABEL, split='test')
    display(pd.DataFrame(test_report['metrics']).T)
    print('Same-audio 45-song control:', test_report['same_audio_heldout_gallery'])
    print('Results:', test_report['output_directory'])
